# 04 - RAG Chain: Build, Test & Deploy

This notebook builds the Retrieval-Augmented Generation chain and deploys it as a Model Serving endpoint.

**Components:**
1. **Retriever**: Databricks Vector Search for semantic similarity
2. **LLM**: `system.ai.claude-haiku-4-5` for answer generation
3. **Chain**: LangChain-based RAG pipeline with MLflow tracing
4. **Deployment**: Registered in MLflow → Deployed to Model Serving

**MLflow Integration:**
- Automatic tracing of all chain invocations
- Model signature for input/output validation
- Registered model in Unity Catalog for governance

In [0]:
%pip install -U databricks-langchain databricks-vectorsearch langchain langchain-core mlflow
dbutils.library.restartPython()

In [0]:
%run ./00_config

In [0]:
# =============================================================================
# MLFLOW SETUP: Experiment & Tracing
# =============================================================================
# Enable MLflow tracing for full observability of the RAG chain.
# Every invocation will be logged with inputs, outputs, and latency.
# =============================================================================

import mlflow

# Set the experiment for tracking
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# Enable MLflow tracing for LangChain
mlflow.langchain.autolog()

print(f"✅ MLflow experiment set: {MLFLOW_EXPERIMENT_NAME}")
print(f"✅ LangChain autologging enabled (tracing active).")

In [0]:
# =============================================================================
# RAG CHAIN: Retriever + LLM with LangChain
# =============================================================================
# Architecture:
#   User Query → Vector Search Retriever → Context + Query → LLM → Answer
#
# The chain:
# 1. Retrieves top-K relevant chunks from Vector Search
# 2. Formats them into a context prompt
# 3. Sends context + user query to Claude Haiku 4.5
# 4. Returns a grounded answer with source attribution
# =============================================================================

from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from databricks.vector_search.client import VectorSearchClient

# --- Initialize Vector Search Retriever ---
vsc = VectorSearchClient()
vs_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

retriever = DatabricksVectorSearch(
    index=vs_index,
    text_column="chunk_to_retrieve",
    columns=["chunk_to_retrieve", "source_file", "chunk_id", "chunk_position"]
).as_retriever(search_kwargs={"k": NUM_RESULTS})

# --- Initialize LLM ---
llm = ChatDatabricks(
    endpoint=LLM_MODEL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)

# --- Define Prompt Template ---
SYSTEM_PROMPT = """You are a knowledgeable assistant for project documentation. 
Answer questions based ONLY on the provided context. If the context does not contain 
relevant information to answer the question, say "I don't have enough information 
in the documentation to answer this question."

Always cite your sources by mentioning the document name.
Be concise, accurate, and helpful.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])


# --- Helper: Format Retrieved Documents ---
def format_docs(docs):
    """Format retrieved documents into a context string with source attribution."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source_file", "Unknown")
        content = doc.page_content
        formatted.append(f"[Source {i}: {source}]\n{content}")
    return "\n\n---\n\n".join(formatted)


# --- Build the RAG Chain ---
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built successfully.")
print(f"   Retriever: {VECTOR_INDEX_NAME} (top-{NUM_RESULTS} chunks)")
print(f"   LLM: {LLM_MODEL} (temp={TEMPERATURE}, max_tokens={MAX_TOKENS})")

In [0]:
# =============================================================================
# TEST: Run Sample Queries
# =============================================================================

test_questions = [
    "What is the Broadcast pipeline and how does it work?",
    "Explain the Fetch Likelihood preprocessing steps.",
    "What is the Hispanic Household Prediction model?"
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"💬 Question: {question}")
    print(f"{'='*60}")
    response = rag_chain.invoke(question)
    print(f"\n🤖 Answer:\n{response}")
    print()

In [0]:
# =============================================================================
# LOG RAG CHAIN TO MLFLOW
# =============================================================================
# Register the chain as an MLflow model for deployment.
# Uses the LangChain flavor with a proper input/output signature.
# =============================================================================

from mlflow.models import infer_signature

# Generate a sample for signature inference
sample_input = "What is the Broadcast pipeline?"
sample_output = rag_chain.invoke(sample_input)

# Infer signature
signature = infer_signature(
    model_input=sample_input,
    model_output=sample_output
)

# Set the Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")

# Log the model
with mlflow.start_run(run_name="rag_chain_v1") as run:
    model_info = mlflow.langchain.log_model(
        lc_model=rag_chain,
        artifact_path="rag_chain",
        signature=signature,
        registered_model_name=RAG_MODEL_NAME,
        input_example=sample_input,
        pip_requirements=[
            "databricks-langchain",
            "databricks-vectorsearch",
            "langchain",
            "langchain-core",
            "mlflow"
        ]
    )
    
    # Log parameters for tracking
    mlflow.log_params({
        "embedding_model": EMBEDDING_MODEL,
        "llm_model": LLM_MODEL,
        "num_retrieval_results": NUM_RESULTS,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "vector_index": VECTOR_INDEX_NAME
    })

print(f"✅ Model logged to MLflow.")
print(f"   Run ID: {run.info.run_id}")
print(f"   Model URI: {model_info.model_uri}")
print(f"   Registered: {RAG_MODEL_NAME}")

In [0]:
# =============================================================================
# DEPLOY TO MODEL SERVING ENDPOINT
# =============================================================================
# Deploy the registered model as a Model Serving endpoint.
# This creates a REST API that the Streamlit app will call.
# =============================================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AutoCaptureConfigInput
)
import time

w = WorkspaceClient()

# Get the latest model version
latest_version = None
for mv in w.model_versions.list(name=RAG_MODEL_NAME):
    if latest_version is None or int(mv.version) > int(latest_version):
        latest_version = mv.version

print(f"   Deploying model version: {latest_version}")

# Create or update the serving endpoint
try:
    endpoint = w.serving_endpoints.get(RAG_CHAIN_ENDPOINT)
    print(f"⏳ Updating existing endpoint '{RAG_CHAIN_ENDPOINT}'...")
    w.serving_endpoints.update_config(
        name=RAG_CHAIN_ENDPOINT,
        served_entities=[
            ServedEntityInput(
                entity_name=RAG_MODEL_NAME,
                entity_version=latest_version,
                scale_to_zero_enabled=True,
                workload_size="Small"
            )
        ]
    )
except Exception:
    print(f"⏳ Creating endpoint '{RAG_CHAIN_ENDPOINT}'...")
    w.serving_endpoints.create(
        name=RAG_CHAIN_ENDPOINT,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    entity_name=RAG_MODEL_NAME,
                    entity_version=latest_version,
                    scale_to_zero_enabled=True,
                    workload_size="Small"
                )
            ],
            auto_capture_config=AutoCaptureConfigInput(
                catalog_name=CATALOG_NAME,
                schema_name=SCHEMA_NAME,
                enabled=True  # Enable inference logging for monitoring
            )
        )
    )

# Wait for endpoint to be ready
print("   Waiting for endpoint to be ready...")
while True:
    ep = w.serving_endpoints.get(RAG_CHAIN_ENDPOINT)
    state = ep.state.ready
    if str(state) == "READY":
        print(f"✅ Endpoint '{RAG_CHAIN_ENDPOINT}' is READY.")
        print(f"   URL: {ep.url if hasattr(ep, 'url') else 'Available in Serving UI'}")
        break
    config_update = ep.state.config_update
    print(f"   State: {state}, Config: {config_update} - waiting...")
    time.sleep(30)

In [0]:
# =============================================================================
# TEST: Query the Deployed Endpoint
# =============================================================================

import json

response = w.serving_endpoints.query(
    name=RAG_CHAIN_ENDPOINT,
    inputs=["Describe the ComScore pipeline process."]
)

print("🚀 Endpoint Test Query: 'Describe the ComScore pipeline process.'")
print(f"\n🤖 Response:")
print(json.dumps(response.as_dict(), indent=2))